# QM9 3D Geometry Utilities Demo

Demonstrates `miniqm.geometry` (`zero_center_of_mass`, `random_rotation`, and `kabsch_align`) on real QM9 molecules from the validation split.

In [ ]:
import pathlib
from typing import Final

import jax
import jax.numpy as jnp
import numpy as np

from miniqm import viz
from miniqm.data import data_source
from miniqm.diffusion import geometry

DATA_DIR: Final = pathlib.Path("../data").resolve()
VALIDATION_PATH: Final = DATA_DIR / "validation.parquet"

source = data_source.DataSource(VALIDATION_PATH, max_molecules=64)
host_batch = next(source.iter_batches(batch_size=16))
sample_batch = host_batch.to_jax()

positions = sample_batch.positions
atom_mask = sample_batch.atom_mask
print(f"positions: {positions.shape}, atom_mask: {atom_mask.shape}")

In [ ]:
shifted = positions + jnp.array([5.0, -3.0, 2.0], dtype=jnp.float32)
recentered = geometry.zero_center_of_mass(shifted, atom_mask)

max_com_err = float(jnp.max(jnp.abs(recentered - positions)))
max_pad_norm = float(jnp.max(jnp.abs(recentered * (~atom_mask)[..., None])))
print(f"Max reconstruction error after shift + zero_center_of_mass: {max_com_err:.2e} A")
print(f"Max padded slot magnitude: {max_pad_norm:.2e} A")

In [ ]:
rotated = geometry.random_rotation(jax.random.key(0), positions, atom_mask)
translated_rotated = (rotated + jnp.array([3.0, -4.0, 1.5], dtype=jnp.float32)) * atom_mask[
    ..., None
]
aligned = geometry.kabsch_align(
    reference=positions,
    target=translated_rotated,
    atom_mask=atom_mask,
)

pair_mask = atom_mask[:, :, None] & atom_mask[:, None, :]
orig_dists = jnp.linalg.norm(positions[:, :, None, :] - positions[:, None, :, :], axis=-1)
rot_dists = jnp.linalg.norm(rotated[:, :, None, :] - rotated[:, None, :, :], axis=-1)
max_dist_diff = float(jnp.max(jnp.abs(orig_dists - rot_dists) * pair_mask))
max_align_err = float(jnp.max(jnp.abs(aligned - positions)))

print(f"Max pairwise distance drift under random_rotation: {max_dist_diff:.2e} A")
print(f"Max Kabsch alignment error on rotated + translated batch: {max_align_err:.2e} A")

In [ ]:
reflected = positions * jnp.array([1.0, 1.0, -1.0], dtype=jnp.float32)
aligned_reflected = geometry.kabsch_align(
    reference=positions,
    target=reflected,
    atom_mask=atom_mask,
)

atom_counts = jnp.sum(atom_mask, axis=-1)
proper_rmsd = jnp.sqrt(jnp.sum((aligned - positions) ** 2, axis=(-1, -2)) / atom_counts)
reflected_rmsd = jnp.sqrt(
    jnp.sum((aligned_reflected - positions) ** 2, axis=(-1, -2)) / atom_counts
)

mol_idx = int(jnp.argmax(reflected_rmsd))
print(
    f"Molecule {mol_idx} ({int(atom_counts[mol_idx])} atoms): "
    f"proper-rotation Kabsch RMSD = {float(proper_rmsd[mol_idx]):.2e} A, "
    f"mirror-enantiomer Kabsch RMSD = {float(reflected_rmsd[mol_idx]):.3f} A"
)
print("3D Overlay: Reference (translucent CPK) vs. Kabsch-Aligned Mirror Enantiomer (cyan):")
viz.show_overlay(
    host_batch.positions[mol_idx],
    np.asarray(aligned_reflected[mol_idx]),
    host_batch.atom_features[mol_idx],
    host_batch.atom_mask[mol_idx],
)